# STAIR training on Kaggle (owner: Hardik)

**Setup (once):**
1. On your laptop: `python -m stair.build_dataset --book tripathi --level section` (or get the folder from Ishaan).
2. Kaggle → *Datasets → New* → upload `stair/data/tripathi/` as a **private** dataset named `stair-data-tripathi`.
3. Upload `stair/train_unsloth.py` as a second private dataset `stair-code` (or paste it into a cell).
4. Notebook settings: **Accelerator = GPU T4**, **Internet = On**, add both datasets as inputs.

**Never** make the dataset, adapter or GGUF public: they are derived from a copyrighted book.

In [ ]:
!pip install -q unsloth
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
DATA = '/kaggle/input/stair-data-tripathi'   # folder with train.jsonl, val.jsonl, labels.json
CODE = '/kaggle/input/stair-code/train_unsloth.py'
OUT  = '/kaggle/working/adapters/tripathi'
import os, json
print(os.listdir(DATA))
print(sum(1 for _ in open(f'{DATA}/train.jsonl')), 'train examples')

## 1. Speed test (≈10 min)
Record **tokens/s** and **peak VRAM** in the team chat — they decide how many epochs fit before 14 Nov.

In [ ]:
!python $CODE --data $DATA --out $OUT --max-steps 100

## 2. Full run + GGUF export
Checkpoints every 200 steps. If the session times out, re-run this cell with `--resume`.

In [ ]:
!python $CODE --data $DATA --out $OUT --epochs 3 --export-gguf

## 3. Bring the model to the demo Mac
Download the `*.gguf` from *Output*, put it in `stair/adapters/tripathi/` and run:

```bash
python -m stair.serve install --book tripathi --gguf stair/adapters/tripathi/<file>.gguf
python -m eval.run_eval retrieval --book tripathi --retrievers vector,router,stair
```